<a href="https://colab.research.google.com/github/anonymous-capybara/platune-bda-demo/blob/main/EMI_Summer2026/audio-diffusion-text2music.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Text-to-Music Generation with Stable Audio Open

**(Colab version)**

This notebook uses the [Stable Audio Open 1.0](https://huggingface.co/stabilityai/stable-audio-open-1.0) model for a basic walkthrough on text-to-music generation. We'll be using [stable-audio-tools](https://github.com/stability-ai/stable-audio-tools) which is a Python library for training and inference audio generative models.




## Setup

<!-- **If you're on your own device**:
 - Open **Terminal**, make sure to create a **new conda environment** (!!), using the following command:  
    - `conda create -n emi-stable-audio python=3.11`  
    - `conda activate emi-stable-audio`
    - `python -m pip install --upgrade pip`
 - We will be using this new `emi-stable-audio` env for this notebook (not the `emi` that we created for the unit)
 - Make sure to select the `emi-stable-audio` env on the top-right corner of VS Code -->

**If you're on Colab**:
 - You're good to go, please run the config below every time you start a new Colab session.


In [ ]:
!git clone https://github.com/Stability-AI/stable-audio-tools.git

In [ ]:
%cd stable-audio-tools

!pip install --upgrade pip
!pip install torch==2.5.1 torchvision==0.20.1 torchaudio==2.5.1 --index-url https://download.pytorch.org/whl/cu124
!pip install alias_free_torch==0.0.6 cached_conv==2.5.0 einops==0.8.1 huggingface_hub librosa==0.11.0 numpy packaging==25.0 safetensors==0.6.2 k-diffusion==0.1.1 einops-exts


## 0. Load Model  

**Important:** We'll be using the HuggingFace **Stable Audio Open 1.0** model, which is a licensed model but free for research purposes. So an important thing to do is to:
 - Go to [HuggingFace](https://huggingface.co/stabilityai/stable-audio-open-1.0) and create an account
 - Go to [Stable Audio Open 1.0](https://huggingface.co/stabilityai/stable-audio-open-1.0) repository, on the right hand side, read and agree its license
 - Then you need to sign in your HuggingFace account here. To do this, first go to generate a HuggingFace **Access Tokens** here: [https://huggingface.co/settings/tokens](https://huggingface.co/settings/tokens).
 - Copy and paste you secret token below:

In [ ]:
## Make sure to hide this token when sending this notebook to someone else
token = "_____your_token_here_______"
!hf auth login --token {token}

Then we can download the model:

In [ ]:
%cd /content/stable-audio-tools

In [ ]:
import torch
import torchaudio
from einops import rearrange
from stable_audio_tools import get_pretrained_model
from stable_audio_tools.inference.generation import generate_diffusion_cond

device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Using device: {device}")

# Download model
model, model_config = get_pretrained_model("stabilityai/stable-audio-open-1.0")
sample_rate = model_config["sample_rate"]

model = model.to(device)

## 1. Text-Conditioned Generation

Stable Audio Open is a **latent diffusion model** that generates audio by iteratively denoising a latent representation, then decoding that back into waveforms.

The model was trained on audio up to 47 seconds at 44.1 kHz stereo. `sample_size` is the output length in `seconds × sample_rate`.

Run the cells below to generate your first clip, then experiment with the prompt and duration.

In [ ]:
# Duration in seconds — change this to control output length
seconds_total = 10
sample_size = seconds_total * sample_rate

# Set up text and timing conditioning
conditioning = [{
    "prompt": "lush orchestral strings, slow, cinematic, emotional",
    "seconds_start": 0,
    "seconds_total": seconds_total
}]

In [ ]:
# Generate stereo audio
output = generate_diffusion_cond(
    model,
    steps=100,
    cfg_scale=7,
    conditioning=conditioning,
    sample_size=sample_size,
    sigma_min=0.3,
    sigma_max=500,
    sampler_type="dpmpp-3m-sde",
    device=device
)

# Rearrange audio batch to a single sequence
output = rearrange(output, "b d n -> d (b n)")

# Peak normalize, clip, convert to int16, and save to file
output = output.to(torch.float32).div(torch.max(torch.abs(output))).clamp(-1, 1).mul(32767).to(torch.int16).cpu()
torchaudio.save("output.wav", output, sample_rate)

In [ ]:
from IPython.display import Audio, display
display(Audio("output.wav"))

## 2. Negative Prompting

Negative prompts add an explicit "unwanted" direction, so the model is pushed away from it at every denoising step.

Use it to suppress specific instruments, timbres, or textures that keep bleeding into your results. `cfg_scale` (classifier free guidance) controls the guidance strength: higher values push harder in both directions (closer to positive, further from negative).

**Try**: generate the same prompt with and without a negative prompt, and compare the results.

In [ ]:
seconds_total = 10
sample_size = seconds_total * sample_rate

conditioning = [{
    "prompt": "lush orchestral strings, slow, cinematic, emotional",
    "seconds_start": 0,
    "seconds_total": seconds_total
}]

negative_conditioning = [{
    "prompt": "drums, percussion, bass, distortion, noise, vocals, lo-fi",
    "seconds_start": 0,
    "seconds_total": seconds_total
}]

output = generate_diffusion_cond(
    model,
    steps=100,
    cfg_scale=7,
    conditioning=conditioning,
    negative_conditioning=negative_conditioning,
    sample_size=sample_size,
    sigma_min=0.3,
    sigma_max=500,
    sampler_type="dpmpp-3m-sde",
    device=device
)

output = rearrange(output, "b d n -> d (b n)")
output = output.to(torch.float32).div(torch.max(torch.abs(output))).clamp(-1, 1).mul(32767).to(torch.int16).cpu()
torchaudio.save("output_negative.wav", output, sample_rate)
print("Saved to output_negative.wav")

In [ ]:
from IPython.display import Audio, display

print("Without negative prompt (output.wav):")
display(Audio("output.wav"))
print("With negative prompt (output_negative.wav):")
display(Audio("output_negative.wav"))

## 3. Prompt Morphing

The model encodes your text prompt into a continuous **embedding vector** in a high-dimensional space. Because this space is smooth, we can the interpolation between two embedding vectors to create a gradual transition from one to another.

At `alpha=0.0` you get Prompt A; at `alpha=1.0` you get Prompt B; values in between blend them proportionally.

In [ ]:
import os

prompt_a = "calm ambient drone, slowly evolving pad, sparse, minimal, quiet"
prompt_b = "jazz guitar, melodic, warm, intimate, clean tone"

seconds_total = 5
sample_size = seconds_total * sample_rate

conditioning_a = [{"prompt": prompt_a, "seconds_start": 0, "seconds_total": seconds_total}]
conditioning_b = [{"prompt": prompt_b, "seconds_start": 0, "seconds_total": seconds_total}]

# Encode both prompts into raw conditioning tensors
with torch.no_grad():
    tensors_a = model.conditioner(conditioning_a, device)
    tensors_b = model.conditioner(conditioning_b, device)

os.makedirs("morphs", exist_ok=True)

alphas = [0.0, 0.25, 0.5, 0.75, 1.0]
for alpha in alphas:
    # Linearly interpolate in embedding space
    blended = {}
    for key in tensors_a:
        ta, tb = tensors_a[key], tensors_b[key]
        if isinstance(ta, (list, tuple)):
            blended[key] = [(1 - alpha) * ta[0] + alpha * tb[0], ta[1]]
        else:
            blended[key] = (1 - alpha) * ta + alpha * tb

    output = generate_diffusion_cond(
        model,
        steps=100,
        cfg_scale=7,
        conditioning_tensors=blended,
        sample_size=sample_size,
        sigma_min=0.3,
        sigma_max=500,
        sampler_type="dpmpp-3m-sde",
        seed=42,    # fixed seed: only the conditioning changes across runs
        device=device
    )

    output = rearrange(output, "b d n -> d (b n)")
    output = output.to(torch.float32).div(torch.max(torch.abs(output))).clamp(-1, 1).mul(32767).to(torch.int16).cpu()
    fname = f"morphs/morph_{alpha:.2f}.wav"
    torchaudio.save(fname, output, sample_rate)
    print(f"  alpha={alpha:.2f} → {fname}")

In [ ]:
from IPython.display import Audio, display

print(f"A: '{prompt_a}'")
print(f"B: '{prompt_b}'\n")
for alpha in alphas:
    print(f"alpha = {alpha:.2f}:")
    display(Audio(f"morphs/morph_{alpha:.2f}.wav"))

## Activities

**Task:** stable-audio-tools gives you a lot more controls for generation. Take a look at its GitHub documentation: [Stable_Audio_Open.md](https://github.com/yukara-ikemiya/friendly-stable-audio-tools/blob/main/docs/Stable_Audio_Open.md), [diffusion.md](https://github.com/yukara-ikemiya/friendly-stable-audio-tools/blob/main/docs/diffusion.md), try to figure out some configuration that you can experiment with. For instance, the Classifier-Free Guidance (CFG) scale (`cfg_scale`), or `init_noise_level` for things like audio-to-audio generation, or audio inpainting.